<a href="https://colab.research.google.com/github/onlyforstudy7770-cpu/DeepFake-voice-detection-project/blob/main/Random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Random Forest is based upon the desision tree

In [ ]:
import pandas as pd
df = pd.read_csv ("500hits.csv" , encoding = 'latin-1')
df.head()

I alredy uploaded the traning and testing data in the google drive so i dont have to wait for dataset to download

In [ ]:
!unzip -q "/content/drive/MyDrive/ASVsfoop_data/AA_lol.zip" -d "/content/my_dataset"

In [ ]:
!unzip -q "/content/drive/MyDrive/ASVsfoop_data/eval_mfcc.zip" -d "/content/my_eval"


In [ ]:
import numpy as np
import pandas as pd
import os
import glob
from tqdm import tqdm

mfcc_folder_path = "/content/my_dataset"
csv_file_path = "/content/drive/MyDrive/ASVsfoop_data/train.csv"

print("Loading CSV...")
df = pd.read_csv(csv_file_path)


df['filename'] = df['filename'].astype(str)
target_map = dict(zip(df["filename"], df["target"]))

# Get list of .npy files
file_paths = glob.glob(os.path.join(mfcc_folder_path, "*.npy"))
file_paths.sort()

X_train_list = []
y_train_list = []

print(f"Found {len(file_paths)} files. Starting data loading...")

# Loop through files with a progress bar
for file_path in tqdm(file_paths):
    full_filename = os.path.basename(file_path)

    # Removing ".flac.npy" to get just the ID (e.g., "LA_T_123")
    csv_key = full_filename.replace(".flac.npy", "").replace(".npy", "")

    # Only load if the file exists in our CSV map
    if csv_key in target_map:
        data = np.load(file_path)

        X_train_list.append(data)
        y_train_list.append(int(target_map[csv_key]))


# Convert lists to numpy arrays
X_train = np.array(X_train_list)
y_train = np.array(y_train_list)

print("\nLoading Complete.")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

Loading CSV...
Found 25380 files. Starting data loading...


100%|██████████| 25380/25380 [00:06<00:00, 3931.95it/s]



Loading Complete.
X_train shape: (25380, 40, 126)
y_train shape: (25380,)


In [ ]:
import numpy as np
import pandas as pd
import os
import glob
from tqdm import tqdm

mfcc_folder_path = "/content/my_eval/eval_mfcc"
csv_file_path = "/content/drive/MyDrive/ASVsfoop_data/eval.csv"

print("Loading CSV...")
df = pd.read_csv(csv_file_path)


df['filename'] = df['filename'].astype(str)
target_map = dict(zip(df["filename"], df["target"]))

file_paths = glob.glob(os.path.join(mfcc_folder_path, "*.npy"))
file_paths.sort()

X_train_list = []
y_train_list = []

print(f"Found {len(file_paths)} files. Starting data loading...")

for file_path in tqdm(file_paths):

    full_filename = os.path.basename(file_path)

    csv_key = full_filename.replace(".flac.npy", "").replace(".npy", "")

    if csv_key in target_map:
        data = np.load(file_path)

        X_train_list.append(data)
        y_train_list.append(int(target_map[csv_key]))


X_test = np.array(X_train_list)
y_test = np.array(y_train_list)

print("\nLoading Complete.")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

Loading CSV...
Found 71237 files. Starting data loading...


100%|██████████| 71237/71237 [00:41<00:00, 1729.15it/s]



Loading Complete.
X_train shape: (16243, 5040)
y_train shape: (16243,)


flatterning the datasets so it can be easy to train

In [ ]:
X_train_2d = X_train.reshape(X_train.shape[0], -1)

In [ ]:
X_test_2d = X_test.reshape(X_test.shape[0], -1)


In [ ]:
print(f"Original shape: {X_train.shape}")
print(f"New shape:      {X_train_2d.shape}")

Original shape: (20304, 40, 126)
New shape:      (20304, 5040)


traning the model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf = RandomForestClassifier(n_estimators=10, class_weight='balanced', random_state=42)

In [ ]:
rf.fit(X_train_2d, y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [ ]:
y_pred = rf.predict(X_test_2d)

In [ ]:
from sklearn.metrics import classification_report , confusion_matrix

In [ ]:
print (classification_report(y_test , y_pred))

              precision    recall  f1-score   support

           0       0.51      0.55      0.53      7355
           1       0.95      0.94      0.94     63882

    accuracy                           0.90     71237
   macro avg       0.73      0.75      0.74     71237
weighted avg       0.90      0.90      0.90     71237



changing the threshold so that we get accurate output

In [ ]:
# Get the probability scores instead of just 0/1
probs = rf.predict_proba(X)

# We are interested in the probability of being Class 1 (Spoof)
probs_spoof = probs[:, 1]

# Custom Threshold: Only call it "Spoof" (1) if model is very sure (> 20%)
# Lowering this threshold makes the model MORE aggressive at catching Spoofs,
threshold = 0.2
y_pred_new = (probs_spoof > threshold).astype(int)

print(classification_report(y, y_pred_new))

              precision    recall  f1-score   support

           0       0.91      0.96      0.93      2580
           1       1.00      0.99      0.99     22800

    accuracy                           0.99     25380
   macro avg       0.95      0.97      0.96     25380
weighted avg       0.99      0.99      0.99     25380



saving the model

In [ ]:
import joblib

model_filename = "/content/drive/MyDrive/ASVsfoop_data/rf_model_99acc.pkl"
joblib.dump(rf, model_filename)

print(f"Model saved to {model_filename}")

Model saved to /content/drive/MyDrive/ASVsfoop_data/rf_model_99acc.pkl
